In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
#import pytensor.tensor as pt

np.random.seed(42)

In [ ]:
# goal:  build a multi-variable Bayesian regression (`Flow ~ Rainfall + Temperature`) and use `pm.MutableData` to predict flow for new weather conditions

# Mock ERA5-like data: River flow depends on rainfall + temperature
n = 60
rainfall = np.random.uniform(5, 80, n)      # mm/day
temperature = np.random.uniform(5, 35, n)    # °C

# i am god; flow inc w/ rain and dec with temp (evapotrans)

TRUE_B_RAIN = 1.2       # m³/s per mm rainfall
TRUE_B_TEMP = -0.3      # m³/s per °C
TRUE_INTERCEPT = 10.0
TRUE_SIGMA = 5.0

flow = TRUE_INTERCEPT + TRUE_B_RAIN * rainfall + TRUE_B_TEMP * temperature + \
       np.random.normal(0, TRUE_SIGMA, n)  # what that slash doing? cant add anything behind it, not even space

In [ ]:
with pm.Model() as hydro_model:

    # mutable data clarification req, why use, when use,
    # so that we can swap these data to train on next set of rain/temp data, while everything remains same


    # pm.data is mutable by default

    rain_data = pm.Data("rain_data", rainfall)
    temp_data = pm.Data("temp_data", temperature)


# intro log normal here cuz rain or flow(intercept) cant be negative
    b_rain = pm.Normal("b_rain", mu=1.0, sigma=2.0)
    b_temp = pm.Normal("b_temp", mu=0.0, sigma=2.0)
    intercept = pm.Normal("intercept", mu=10, sigma=20)
    sigma = pm.HalfNormal("sigma", sigma=10)

    expected_flow = pm.Deterministic("expected_flow", intercept + b_rain * rain_data + b_temp * temp_data)

    # as exp flow is neg, pos approaches to 0, if we use mod, a drought gonna magically be a flood

    # to upgrade the model and have strictly positive values, we use gamma and then shrink priors to match the log/exp scale but for now we gauss

   # expected_flow_pos = pt.exp(expected_flow)

   # also alt parameterization, behind the bb, pymc auto converts mu and sigma passing into alpha and beta if gamma used

# same change distro
    y = pm.Normal("y", mu=expected_flow, sigma =sigma, observed=flow)

    prior_checks = pm.sample_prior_predictive(samples=2000, random_seed=42)

    trace = pm.sample(2000, tune=1000, cores=2, chains=2, random_seed=42, progressbar=False)
    ppc = pm.sample_posterior_predictive(trace=trace, random_seed=42, progressbar=False)



fig, ax = plt.subplots(figsize=(12, 6))
az.plot_ppc(prior_checks,
group="prior",
kind="kde",
ax=ax,
colors=['gray', 'black', 'blue'], alpha=0.8)
az.plot_kde(flow, ax=ax, plot_kwargs={"color": "red", "linewidth": 3, "linestyle": "--"}, label="Actual Observed Data")
plt.title("Prior Predictive Check: Physicality Audit")
plt.tight_layout()
plt.show()


print(az.summary(trace, var_names=["b_rain", "b_temp", "intercept", "sigma"]))
az.plot_ppc(ppc, observed_rug=True)
plt.title("Posterior Predictive Check: Flow Model")
plt.tight_layout()
plt.show()
display(pm.model_to_graphviz(hydro_model))


